# Local LLM → Generate Dialogue → Create HTML
**ECON 422 · Macroeconomics & Machine Learning**

Run this notebook to generate an original Elizabeth Bennet–Sherlock Holmes conversation with a model running **inside Python on your own computer**, then create the same click-through HTML display.

No Gemini account, API key, paid inference service, or local server is used. Both characters share one model but receive different role instructions. Internet is needed to install packages and download the model once. Generation then runs locally.

Use Python 3.10 or newer. The default model is **Qwen2.5-1.5B-Instruct**. Its weights take roughly 3 GB in half precision; loading and generation need additional memory (CPU float32 weights take roughly 6 GB). For a smaller model, change the model ID to `Qwen/Qwen2.5-0.5B-Instruct`. CPU generation can be slow. Start with two rounds if desired.

Run cells in order. If packages were already imported before installation, restart the kernel afterward. Skip the installation cell when working offline.

## What this notebook outputs

After you run all cells, the `outputs` folder will contain:

- `elizabeth_sherlock_dialogue.csv`
- `elizabeth_sherlock_dialogue.json`
- `elizabeth_sherlock_dialogue.txt`
- `elizabeth_sherlock_click_dialogue.html`

The HTML is the final visual output.


In [ ]:
%pip install -q "torch>=2.2,<3" "transformers==4.57.1" "accelerate>=0.26,<2" "pandas>=2,<3"

## 1. Choose the local model

The first run saves model files under `local_models` beside the notebook's working directory. After that, the notebook loads only that folder. Set `ALLOW_MODEL_DOWNLOAD = False` to require an existing local copy.

`DEVICE = "auto"` uses CUDA, Apple Silicon MPS, or CPU in that order. Set it to `"cpu"` if your GPU backend has a compatibility problem. Changing the model ID uses a different local folder.

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ALLOW_MODEL_DOWNLOAD = True  # Needed only to obtain a missing model.
DEVICE = "auto"             # "auto", "cpu", "mps", or "cuda"
SEED = 422
TEMPERATURE = 0.8
MAX_NEW_TOKENS = 280
MODEL_DIR = Path("local_models") / MODEL_ID.split("/")[-1]
print("Local model folder:", MODEL_DIR.resolve())

## 2. Load the model into local memory

In [ ]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import pandas as pd
import json
import base64
import mimetypes
import html as html_lib
import time
from IPython.display import display, FileLink

if DEVICE == "auto":
    device = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu")
else:
    device = DEVICE
if device not in {"cpu", "cuda", "mps"}:
    raise ValueError("DEVICE must be auto, cpu, cuda, or mps")
if device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Choose DEVICE='cpu'.")
if device == "mps" and not torch.backends.mps.is_available():
    raise RuntimeError("MPS is unavailable. Choose DEVICE='cpu'.")
dtype = torch.float32 if device == "cpu" else torch.float16
ready = MODEL_DIR / ".download_complete"
if ready.exists():
    source = str(MODEL_DIR.resolve())
    offline = True
    print("Loading saved model (offline)...")
else:
    if not ALLOW_MODEL_DOWNLOAD:
        raise FileNotFoundError(f"No complete model at {MODEL_DIR.resolve()}. Enable download once.")
    source = MODEL_ID
    offline = False
    print("Downloading model files once; this may take several minutes...")

# These load model weights, not remotely generated answers.
tokenizer = AutoTokenizer.from_pretrained(source, local_files_only=offline, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    source, local_files_only=offline, trust_remote_code=False,
    torch_dtype=dtype, use_safetensors=True, low_cpu_mem_usage=True,
)
if not offline:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MODEL_DIR)
    ready.write_text(MODEL_ID, encoding="utf-8")
model = model.to(device).eval()
set_seed(SEED)
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
print("Model:", MODEL_ID)
print("Device:", device, "| dtype:", dtype)
print("Output directory:", output_dir.resolve())
print("Model loaded. Dialogue generation now runs entirely inside Python.")

## 3. Choose the topic and number of rounds

In [ ]:
TOPICS = {
    "first_impressions": "How much can you really know about a stranger in five minutes?",
    "love_vs_logic": "When should a person trust emotion over reason?",
    "manners_vs_truth": "Is politeness ever more valuable than telling the truth?",
    "evidence_vs_intuition": "When evidence and intuition disagree, which deserves more trust?",
    "marriage_and_compatibility": "Can compatibility be reasoned out, or must it be discovered emotionally?"
}

topic = TOPICS["first_impressions"]
N_ROUNDS = 6

print("Topic:", topic)
print("Rounds:", N_ROUNDS)


## 4. Character instructions

In [ ]:
ELIZABETH_ROLE = """
You are Elizabeth Bennet from Pride and Prejudice.

Core personality:
- intelligent, lively, witty, and socially perceptive
- skeptical of pretension and status
- playful and sharp, but capable of self-correction
- attentive to character, dignity, affection, and moral judgment

Write ORIGINAL dialogue only.
Do not quote or closely reproduce the novel or film.
Keep each turn to 2-5 sentences.
Respond directly to Sherlock Holmes.
"""

HOLMES_ROLE = """
You are Sherlock Holmes.

Core personality:
- intensely observant, analytical, precise, and skeptical
- values evidence, inference, and disciplined reasoning
- dryly witty and sometimes overconfident
- interested in human behavior as evidence

Write ORIGINAL dialogue only.
Do not quote or closely reproduce books, film, or television.
Keep each turn to 2-5 sentences.
Respond directly to Elizabeth Bennet.
"""


## 5. Local generation helper

In [ ]:
def ask_agent(role, message, max_output_tokens=MAX_NEW_TOKENS):
    messages = [
        {"role": "system", "content": role + "\nReturn only your character's spoken reply, without a speaker label."},
        {"role": "user", "content": message},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    context_limit = getattr(model.config, "max_position_embeddings", 32768)
    if inputs["input_ids"].shape[1] + max_output_tokens > context_limit:
        raise ValueError("Prompt exceeds the model context window; shorten the topic or instructions.")
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=max_output_tokens, do_sample=True,
            temperature=TEMPERATURE, top_p=0.9, repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode the new answer only; omit the original prompt.
    answer_ids = generated[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    if not answer:
        raise RuntimeError("The local model returned an empty reply. Try another seed or a larger token limit.")
    return answer

## 6. Generate the dialogue

The two roles take alternating turns using the loaded local model. The original conversation structure is preserved. A progress line appears after each completed turn; the first turn can take longer.

Each completed turn is also saved in `outputs/dialogue_in_progress.json`. Rerunning this cell starts a new conversation and overwrites that checkpoint. The final output filenames also replace the previous run.

In [ ]:
if not isinstance(N_ROUNDS, int) or N_ROUNDS < 1:
    raise ValueError("N_ROUNDS must be a positive integer")
started = time.monotonic()
def checkpoint(rows):
    (output_dir / "dialogue_in_progress.json").write_text(
        json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Progress: {len(rows)}/{2*N_ROUNDS} turns | {time.monotonic()-started:.1f}s elapsed", flush=True)

conversation = []
turn = 1

message_for_elizabeth = f"""
You have just met Sherlock Holmes.

Conversation topic:
{topic}

Open the conversation with a sharp, engaging thought that reveals your personality.
"""

for round_no in range(1, N_ROUNDS + 1):
    elizabeth = ask_agent(ELIZABETH_ROLE, message_for_elizabeth)
    conversation.append({
        "turn": turn,
        "speaker": "Elizabeth Bennet",
        "text": elizabeth
    })
    print(f"Turn {turn} | Elizabeth Bennet\n{elizabeth}\n")
    checkpoint(conversation)
    turn += 1

    message_for_holmes = f"""
Conversation topic:
{topic}

Elizabeth Bennet has just said:
{elizabeth}

Respond directly to her. Let your method of observation and deduction clash productively
with her social and moral judgment. Move the conversation forward.
"""
    holmes = ask_agent(HOLMES_ROLE, message_for_holmes)
    conversation.append({
        "turn": turn,
        "speaker": "Sherlock Holmes",
        "text": holmes
    })
    print(f"Turn {turn} | Sherlock Holmes\n{holmes}\n")
    checkpoint(conversation)
    turn += 1

    message_for_elizabeth = f"""
Conversation topic:
{topic}

Your previous remark:
{elizabeth}

Sherlock Holmes replied:
{holmes}

Answer him directly. Challenge him where appropriate, concede something if warranted,
and let your personality become clearer through the disagreement.
"""

print("Dialogue generation complete.")


## 7. Save the dialogue

In [ ]:
df = pd.DataFrame(conversation)
display(df)

df.to_csv(output_dir / "elizabeth_sherlock_dialogue.csv", index=False)
df.to_json(output_dir / "elizabeth_sherlock_dialogue.json",
           orient="records", indent=2, force_ascii=False)

with open(output_dir / "elizabeth_sherlock_dialogue.txt", "w", encoding="utf-8") as f:
    f.write(f"TOPIC: {topic}\n\n")
    for row in conversation:
        f.write(f"Turn {row['turn']} | {row['speaker']}\n")
        f.write(row["text"] + "\n\n")

print("Saved CSV, JSON, and TXT files.")


## 8. Build the HTML visualization automatically

If available, place `elizabeth_bennet_pride_prejudice_user_image.png` and `sherlock_user_image.png` in the notebook's working directory. They are embedded in the HTML. Missing images are omitted automatically; character names and the dialogue controls still work.

In [ ]:
# Read bundled images and embed them directly into the HTML
elizabeth_img = Path("elizabeth_bennet_pride_prejudice_user_image.png")
sherlock_img = Path("sherlock_user_image.png")

def image_to_data_uri(path):
    if not path.is_file():
        print("Optional portrait missing:", path.name)
        return ""
    mime = mimetypes.guess_type(path.name)[0] or "image/png"
    return "data:" + mime + ";base64," + base64.b64encode(path.read_bytes()).decode("ascii")

elizabeth_src = image_to_data_uri(elizabeth_img)
sherlock_src = image_to_data_uri(sherlock_img)

# Prevent generated text from closing the HTML script element.
dialogue_json = json.dumps(conversation, ensure_ascii=False).replace("<", "\\u003c").replace(">", "\\u003e").replace("&", "\\u0026")
safe_topic = html_lib.escape(topic)
elizabeth_style = "" if elizabeth_src else "display:none"
sherlock_style = "" if sherlock_src else "display:none"

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Elizabeth Bennet Meets Sherlock Holmes</title>
<style>
:root {{
  --warm:#f5eee4;
  --cool:#eaf1f5;
  --ink:#20262c;
  --muted:#69727a;
  --liz:#9c6c45;
  --holmes:#425f79;
}}
*{{box-sizing:border-box}}
body{{
  margin:0;
  min-height:100vh;
  color:var(--ink);
  background:linear-gradient(135deg,var(--warm),var(--cool));
  font-family:Georgia,"Times New Roman",serif;
}}
.page{{max-width:1240px;margin:auto;padding:28px 20px 42px}}
h1{{text-align:center;margin:0;font-size:clamp(1.8rem,4vw,2.8rem)}}
.subtitle{{text-align:center;color:var(--muted);max-width:850px;margin:10px auto 20px;line-height:1.5}}
.topic{{width:min(860px,100%);margin:0 auto 24px;padding:13px 18px;border-radius:999px;text-align:center;background:rgba(255,255,255,.75);box-shadow:0 8px 18px rgba(0,0,0,.07)}}
.stage{{display:grid;grid-template-columns:minmax(220px,1fr) minmax(380px,1.45fr) minmax(220px,1fr);gap:18px;align-items:stretch}}
.character{{background:rgba(255,255,255,.88);border-radius:26px;padding:16px;box-shadow:0 14px 34px rgba(0,0,0,.11);border:3px solid transparent;transition:.22s ease;display:flex;flex-direction:column;align-items:center}}
.character.active{{transform:translateY(-7px) scale(1.01)}}
#lizCard.active{{border-color:var(--liz)}}
#holmesCard.active{{border-color:var(--holmes)}}
.photo{{width:100%;aspect-ratio:4/5;object-fit:cover;border-radius:18px;background:#ddd}}
.name{{font-size:1.35rem;font-weight:700;margin-top:14px;text-align:center}}
.actor{{margin-top:5px;color:var(--muted);text-align:center;line-height:1.35}}
.work{{font-size:.92rem;color:var(--muted);text-align:center;margin-top:4px}}
.dialogue-panel{{min-height:570px;background:rgba(255,255,255,.8);border-radius:28px;padding:22px;box-shadow:0 14px 34px rgba(0,0,0,.11);display:flex;flex-direction:column}}
.toprow{{display:flex;justify-content:space-between;color:var(--muted);font-size:.94rem;gap:12px}}
.speech-zone{{flex:1;display:flex;align-items:center;padding:20px 0;cursor:pointer;user-select:none}}
.bubble-row{{width:100%;display:flex}}
.bubble{{max-width:88%;padding:21px 22px;border-radius:25px;background:#fff;box-shadow:0 9px 22px rgba(0,0,0,.08);font-size:1.12rem;line-height:1.65}}
.bubble.left{{margin-right:auto;border-left:6px solid var(--liz)}}
.bubble.right{{margin-left:auto;border-right:6px solid var(--holmes)}}
.speaker{{text-transform:uppercase;letter-spacing:.06em;font-size:.82rem;color:var(--muted);margin-bottom:9px;font-family:Arial,sans-serif;font-weight:700}}
.controls{{display:flex;gap:10px;justify-content:center;flex-wrap:wrap}}
button{{border:0;border-radius:999px;padding:12px 18px;cursor:pointer;font-size:.96rem;box-shadow:0 6px 16px rgba(0,0,0,.1)}}
#next{{background:#20262c;color:#fff}}
#prev,#restart{{background:#fff;color:#20262c}}
.hint{{text-align:center;color:var(--muted);font-size:.9rem;margin-top:11px}}
.footer{{margin-top:22px;font-family:Arial,sans-serif;font-size:.78rem;color:var(--muted);line-height:1.45;text-align:center}}
@media(max-width:900px){{
  .stage{{grid-template-columns:1fr}}
  .dialogue-panel{{order:-1;min-height:410px}}
  .character{{max-width:560px;width:100%;margin:auto}}
  .photo{{aspect-ratio:3/2}}
  .bubble{{max-width:100%}}
}}
</style>
</head>
<body>
<div class="page">
  <h1>Elizabeth Bennet Meets Sherlock Holmes</h1>
  <div class="subtitle">
    A personality-revealing conversation between two very different readers of human behavior.
  </div>
  <div class="topic"><strong>Topic:</strong> {safe_topic}</div>

  <div class="stage">
    <section id="lizCard" class="character">
      <img class="photo" style="{elizabeth_style}" src="{elizabeth_src}" alt="Elizabeth Bennet image">
      <div class="name">Elizabeth Bennet</div>
      <div class="actor">Witty, socially perceptive, and independent</div>
      <div class="work"><em>Pride &amp; Prejudice</em> (2005)</div>
    </section>

    <section class="dialogue-panel">
      <div class="toprow">
        <div id="counter"></div>
        <div>Click the dialogue to continue</div>
      </div>
      <div id="speechZone" class="speech-zone">
        <div class="bubble-row">
          <div id="bubble" class="bubble left">
            <div id="speaker" class="speaker"></div>
            <div id="line"></div>
          </div>
        </div>
      </div>
      <div class="controls">
        <button id="prev">Previous line</button>
        <button id="next">Next line</button>
        <button id="restart">Restart</button>
      </div>
      <div class="hint">The portrait border highlights the current speaker.</div>
    </section>

    <section id="holmesCard" class="character">
      <img class="photo" style="{sherlock_style}" src="{sherlock_src}" alt="Sherlock Holmes image">
      <div class="name">Sherlock Holmes</div>
      <div class="actor">Observant, analytical, and skeptical</div>
      <div class="work">221B Baker Street</div>
    </section>
  </div>

  <div class="footer">
    Original hypothetical dialogue generated locally in Jupyter with {html_lib.escape(MODEL_ID)}.
  </div>
</div>

<script>
const dialogue = {dialogue_json};
let i = 0;
const lizCard = document.getElementById("lizCard");
const holmesCard = document.getElementById("holmesCard");
const bubble = document.getElementById("bubble");
const speaker = document.getElementById("speaker");
const line = document.getElementById("line");
const counter = document.getElementById("counter");
const speechZone = document.getElementById("speechZone");

function render() {{
  const d = dialogue[i];
  const liz = d.speaker === "Elizabeth Bennet";
  speaker.textContent = d.speaker;
  line.textContent = d.text;
  counter.textContent = `Turn ${{i+1}} of ${{dialogue.length}}`;
  bubble.className = "bubble " + (liz ? "left" : "right");
  lizCard.classList.toggle("active", liz);
  holmesCard.classList.toggle("active", !liz);
}}
function nextLine() {{ i = (i + 1) % dialogue.length; render(); }}
function prevLine() {{ i = (i - 1 + dialogue.length) % dialogue.length; render(); }}
function restart() {{ i = 0; render(); }}

document.getElementById("next").onclick = nextLine;
document.getElementById("prev").onclick = prevLine;
document.getElementById("restart").onclick = restart;
speechZone.onclick = nextLine;
render();
</script>
</body>
</html>
"""

html_path = output_dir / "elizabeth_sherlock_click_dialogue.html"
html_path.write_text(html, encoding="utf-8")
print("Created HTML:", html_path.resolve())

display(FileLink(str(html_path)))


## 9. Open your dialogue

Click the HTML link above, or open `outputs/elizabeth_sherlock_click_dialogue.html`. It is standalone and works offline. CSV, JSON, and TXT copies remain in the same folder.

The model weights are downloaded separately, not bundled in this notebook. Local generation was not executed in the editing environment because the model and PyTorch were unavailable. The notebook structure and export logic were checked.

Technical references: [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) and [Transformers installation and offline use](https://huggingface.co/docs/transformers/v4.57.1/installation).